<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 3 · STATISTICAL REASONING FOR BUSINESS</span>

# Session 3: t-test or chi-square? Choosing the right test

**Data:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). `ev_m3_sample.csv` is the cleaned 10,000-row class sample from Module 2. `ev_m3_population.csv` has 50,000 rows that we treat as **everybody**, so we can see what sampling does.
**Time:** 60 minutes. **Course outcome:** CO3.

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | Three manager questions: which test? |
| Concept | 10 | The which-test rule (instructor) |
| Guided lab | 30 | t-test, chi-square, ANOVA, Fisher exact on the EV data |
| Team task | 10 | Choose and run the test for four questions |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- pick the test from the *type* of the two columns
- run a Welch t-test, chi-square, ANOVA and Fisher exact test and read the output
- check the assumption behind a test before trusting its p-value
- use a test to check whether a pattern survives a lurking variable

**New commands this session (max 5):** `stats.ttest_ind(equal_var=False)`, `pd.crosstab`, `stats.chi2_contingency`, `stats.f_oneway`, `stats.fisher_exact`

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [ ]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# -- Load the data -------------------------------------------------------------
# Needs two files: ev_m3_sample.csv (10,000 rows: the cleaned Module 2 sample)
#                  ev_m3_population.csv (50,000 rows: we treat these as "everybody")
def find(name):
    for folder in [".", "data", "../data"]:
        if os.path.exists(os.path.join(folder, name)):
            return os.path.join(folder, name)
    try:
        from google.colab import files
        print("Choose", name); files.upload(); return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} next to this notebook or in a data folder.")

def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

df = add_derived(pd.read_csv(find("ev_m3_sample.csv")))        # the class sample
pop = add_derived(pd.read_csv(find("ev_m3_population.csv")))   # "everybody"
print("Sample:", len(df), "rows | Population:", len(pop), "rows")

## Part A. Hook: which test? (5 min)
Before running anything, write the test you would use for each question (t-test, chi-square, or ANOVA).
1. *Do buyers earn more than non-buyers?*  
2. *Is city type related to buying?*  
3. *Does average income differ across the three city types?*

> _write your three answers here_

## Part B. Check an assumption: the Q-Q plot (5 min)
A t-test works best when averages are bell-shaped. A Q-Q plot compares our data with a bell curve: dots on the line mean 'close to bell-shaped'.

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 3.8))
stats.probplot(df["Annual_Income_USD"], plot=ax)
ax.get_lines()[0].set(color=C["lagoon"], markersize=3); ax.get_lines()[1].set(color=C["terracotta"])
ax.set_title("Q-Q plot of income")
plt.tight_layout()

**YOUR ANSWER:** Where do the dots leave the line? Which feature of the income data from Module 2 explains it? With 10,000 rows, would you still run a t-test?

> _write here_

## Part C. t-test: number by two groups (5 min)
**Predict first:** will the t-statistic for income be bigger or smaller than for age? (A big t means a big gap relative to the noise.)

In [ ]:
yes = df[df["buy"] == 1]
no = df[df["buy"] == 0]

res = stats.ttest_ind(yes["Annual_Income_USD"], no["Annual_Income_USD"], equal_var=____)   # TODO: Welch test = do NOT assume equal spreads (True or False?)
print(f"Average income: buyers {yes['Annual_Income_USD'].mean():,.0f} vs non-buyers {no['Annual_Income_USD'].mean():,.0f}")
print(f"t = {res.statistic:.1f}, p = {res.pvalue:.2g}")

In [ ]:
rows = []
for col in ["Age", "Annual_Income_USD", "Daily_Commute_km"]:
    r = stats.ttest_ind(yes[col], no[col], equal_var=False)
    rows.append({"column": col, "buyers_mean": yes[col].mean(), "non_buyers_mean": no[col].mean(), "t": r.statistic, "p": r.pvalue})
pd.DataFrame(rows).round(3)

**YOUR ANSWER:** Which columns differ between buyers and non-buyers? Compare with your shuffle-test results from Session 2. Do the two methods agree?

> _write here_

## Part D. Chi-square: category by category (5 min)
**Predict first:** is city type related to buying? Roughly how strong?

In [ ]:
table = pd.crosstab(df["City_Type"], df["buy"])                  # counts: rows = city, columns = 0/1
chi2, p, dof, expected = stats.____(table)                      # TODO: the chi-square test of a table (chi2_contingency)
print(table, "\n")
print(pd.crosstab(df["City_Type"], df["buy"], normalize="index").round(3), "\n")      # Yes rate per city
print(f"chi-square = {chi2:.1f}, degrees of freedom = {dof}, p = {p:.2g}")
print("Smallest expected count:", expected.min().round(0), "(should be at least 5)")

**YOUR ANSWER:** Is city type related to buying? Does the chi-square tell you *which* city is higher? (Look at the second table.)

> _write here_

## Part E. ANOVA: a number across three groups (5 min)
Does average **income** differ across Rural, Suburban and Urban? **Predict first.**

In [ ]:
groups = [g["Annual_Income_USD"] for _, g in df.groupby("City_Type")]
print(df.groupby("City_Type")["Annual_Income_USD"].mean().round(0), "\n")
f = stats.____(*groups)                                         # TODO: one-way ANOVA (f_oneway); the * unpacks the three groups
print(f"F = {f.statistic:.2f}, p = {f.pvalue:.2f}")

**YOUR ANSWER:** Does income differ by city? Why is that useful to know before we read the city pattern in buying?

> _write here_

## Part F. Does the city pattern survive a lurking variable? (5 min)
In Module 2 we saw home charging hiding behind city type. Test city vs buying **inside** each home-charging group.

In [ ]:
for h in ["Yes", "No"]:
    sub = df[df["Home_Charging_Possible"] == h]
    t = pd.crosstab(sub["City_Type"], sub["buy"])
    chi2, p, dof, exp = stats.chi2_contingency(t)
    rates = sub.groupby("City_Type")["buy"].mean().round(3).to_dict()
    print(f"Home charging = {h}: chi-square = {chi2:.1f}, p = {p:.3f}, smallest expected count = {exp.min():.0f}")
    print("   Yes rates:", rates)

**YOUR ANSWER:** Compare the two groups. In which group is the city difference still significant? What does this tell you about *why* rural looked best in Module 2?

> _write here_

## Part G. A small group: Fisher exact test (5 min)
In **rural** areas only, how few people lack home charging? Compare buying rates with and without home charging.

In [ ]:
rural = df[df["City_Type"] == "Rural"]
t = pd.crosstab(rural["Home_Charging_Possible"], rural["buy"])
print(t, "\n")
print(f"Yes rate without home charging: {t.loc['No', 1] / t.loc['No'].sum():.1%}   with: {t.loc['Yes', 1] / t.loc['Yes'].sum():.1%}")
print("Chi-square p =", round(stats.chi2_contingency(t)[1], 3), "  smallest expected count =", stats.chi2_contingency(t)[3].min().round(1))
odds, p_fisher = stats.____(t)                                  # TODO: exact test for a 2x2 table (fisher_exact)
print("Fisher exact p =", round(p_fisher, 3))

**YOUR ANSWER:** The group without home charging is tiny. Would you quote its percentage to a manager without a warning? What extra thing would you report? (Session 4 gives you the tool.)

> _write here_

## Team task (10 min): choose, run, decide
Fill the table. For each question, name the two column types, pick the test, run it, and write a one-line decision. Use `yes`, `no`, `df`, `pd.crosstab`, `stats.ttest_ind` and `stats.chi2_contingency`.

| # | Question | Column types | Test | p-value | Decision |
|---|---|---|---|---|---|
| 1 | Does a subsidy go with buying? | | | | |
| 2 | Does environmental concern (1-5) go with buying? | | | | |
| 3 | Does gender go with buying? | | | | |
| 4 | Do buyers own more cars? | | | | |
| 5 | Does range anxiety (Low/Medium/High) go with buying? Check expected counts. | | | | |

## Exit ticket (5 min)
1. You compare average income across four regions. Which test?
2. A chi-square table has an expected count of 2. What do you do?
3. Why do we use `equal_var=False` by default?